In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

In [2]:
EXPORT_DIR = "data/facteurs_systemiq"
res_cons = pd.read_csv(f"{EXPORT_DIR}/resid_construction.csv")
res_ind = pd.read_csv(f"{EXPORT_DIR}/resid_industrie.csv")
res_ext = pd.read_csv(f"{EXPORT_DIR}/resid_extraction_primaire.csv")
res_fin = pd.read_csv(f"{EXPORT_DIR}/resid_finance.csv")
res_info = pd.read_csv(f"{EXPORT_DIR}/resid_info_com.csv")
res_public = pd.read_csv(f"{EXPORT_DIR}/resid_public.csv")
res_serv_pro = pd.read_csv(f"{EXPORT_DIR}/resid_services_pro.csv")
res_serv = pd.read_csv(f"{EXPORT_DIR}/resid_services.csv")

In [3]:
res_ind

,date,0
0,2011-07-01,-0.098782
1,2012-01-01,0.145020
2,2012-04-01,-0.554485
3,2012-07-01,-0.906841
4,2012-10-01,-0.034266
5,2013-01-01,0.125134
6,2013-04-01,-0.577032
7,2013-07-01,0.319064
8,2013-10-01,0.014697
9,2014-01-01,-0.020362


In [4]:
import pandas as pd
from functools import reduce

dfs_list = [res_cons, res_ind, res_ext, res_fin, res_info, res_public, res_serv_pro, res_serv ]
names = ["construction", "industrie", "extraction_primaire", "finance", "info_com", "public", "services_pro", "services"]

import pandas as pd
from functools import reduce

dfs_renamed = []

for name, df in zip(names, dfs_list):
    tmp = df.copy()

    # date en datetime
    tmp["date"] = pd.to_datetime(tmp["date"])

    # toutes les colonnes sauf date
    value_cols = [c for c in tmp.columns if c != "date"]

    if len(value_cols) == 1:
        # 1 seule colonne valeur -> elle devient "name"
        tmp = tmp.rename(columns={value_cols[0]: name})
    else:
        # plusieurs colonnes valeur -> "name_col"
        tmp = tmp.rename(columns={c: f"{name}_{c}" for c in value_cols})

    dfs_renamed.append(tmp)

df_merged = reduce(lambda l, r: pd.merge(l, r, on="date", how="outer"), dfs_renamed)
df_merged = df_merged.sort_values("date").reset_index(drop=True)

In [5]:
df_merged

,date,construction,industrie,extraction_primaire,finance,info_com,public,services_pro,services
0,2011-07-01,NaN,-0.098782,-0.226968,NaN,NaN,0.862419,-1.323745,NaN
1,2011-10-01,-0.664780,NaN,0.532706,-1.058971,-0.333045,1.056578,NaN,-0.367661
2,2012-01-01,NaN,0.145020,0.875752,NaN,NaN,NaN,0.359605,NaN
3,2012-04-01,-0.304486,-0.554485,-0.214829,0.671036,0.568009,-0.660016,0.706535,1.188916
4,2012-07-01,-1.076204,-0.906841,-0.425546,0.460393,-0.731019,NaN,0.326652,-0.342172
5,2012-10-01,-1.298520,-0.034266,-0.346174,0.414796,0.967104,0.890116,0.287050,-0.142330
6,2013-01-01,-0.521909,0.125134,-0.276526,-0.083916,1.133157,1.087691,0.391640,0.743777
7,2013-04-01,-1.319508,-0.577032,-0.187663,0.130548,0.536248,1.116949,-0.817081,0.720985
8,2013-07-01,-0.127279,0.319064,0.087433,0.616297,-0.286776,-0.030680,1.514095,0.133188
9,2013-10-01,-0.504486,0.014697,0.041243,-0.265709,0.416125,0.373277,0.275970,-0.248984


In [6]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

X = df_merged.drop(columns=['date'])
cols = X.columns
m = len(cols)

BP_star = 0.0

for a in range(m):
    for b in range(a+1, m):
        xi = X.iloc[:, a]
        xj = X.iloc[:, b]
        mask = xi.notna() & xj.notna()
        Tij = int(mask.sum())
        if Tij >= 3:  # minimum pratique pour une corrélation
            rij = np.corrcoef(xi[mask], xj[mask])[0, 1]
            BP_star += Tij * (rij**2)

df_chi2 = m * (m - 1) // 2
p_value_star = 1 - chi2.cdf(BP_star, df_chi2)

BP_star, df_chi2, p_value_star

(np.float64(79.46050328052138), 28, np.float64(8.029370587303575e-07))

In [7]:
df_merged= df_merged.rename(columns={
    'construction': 'construction',
    'industrie': 'industry',
    'extraction_primaire': 'primary_extraction',
    'finance': 'finance',
    'info_com': 'information_communication',
    'public': 'public_sector',
    'services_pro': 'business_services',
    'services': 'services'
})

# On enlève la colonne date
X = df_merged.drop(columns=['date'])


# Matrice de corrélation (pairwise deletion par défaut)
R = X.corr()

R

,construction,industry,primary_extraction,finance,information_communication,public_sector,business_services,services
construction,1.000000,0.073858,0.001569,0.172753,-0.077176,-0.235260,-0.201819,0.040388
industry,0.073858,1.000000,0.247563,0.191033,0.403262,0.187893,0.464311,0.306667
primary_extraction,0.001569,0.247563,1.000000,0.045639,0.252900,0.143582,0.354251,0.201599
finance,0.172753,0.191033,0.045639,1.000000,0.246546,0.132180,0.320781,0.616437
information_communication,-0.077176,0.403262,0.252900,0.246546,1.000000,0.272595,0.419138,0.333760
public_sector,-0.235260,0.187893,0.143582,0.132180,0.272595,1.000000,0.120035,0.158309
business_services,-0.201819,0.464311,0.354251,0.320781,0.419138,0.120035,1.000000,0.250210
services,0.040388,0.306667,0.201599,0.616437,0.333760,0.158309,0.250210,1.000000


In [8]:
def breusch_pagan_contemp(resid_df):
    """
    resid_df : DataFrame (T x m) de résidus par secteur (alignés sur un index temps).
    Le test est fait sur l'intersection des dates (lignes complètes).
    """
    E = resid_df.dropna(axis=0, how="any")  # intersection
    T, m = E.shape
    if m < 2:
        raise ValueError("Il faut au moins 2 secteurs (m>=2).")
    if T < 3:
        raise ValueError("Pas assez d'observations communes pour calculer les corrélations.")

    R = E.corr()

    # Somme des r_ij^2 pour i<j
    r2_sum = 0.0
    cols = list(R.columns)
    for i in range(m):
        for j in range(i + 1, m):
            rij = R.loc[cols[i], cols[j]]
            r2_sum += float(rij**2)

    BP = T * r2_sum
    df_chi = m * (m - 1) / 2
    pval = 1 - chi2.cdf(BP, df_chi)

    # Top corrélations (utile pour diagnostiquer)
    R_off = R.copy()
    np.fill_diagonal(R_off.values, np.nan)
    top_pairs = (
        R_off.abs()
             .stack()
             .sort_values(ascending=False)
             .head(15)
    )

    return {
        "T_common": T,
        "m": m,
        "BP": BP,
        "df": df_chi,
        "pvalue": pval,
        "R": R,
        "top_abs_corr_pairs": top_pairs,
        "E_common": E
    }

In [9]:
breusch_pagan_contemp(df_merged.drop(columns="date"))["R"]

,construction,industry,primary_extraction,finance,information_communication,public_sector,business_services,services
construction,1.000000,0.127473,-0.161388,0.194279,-0.162352,-0.192459,-0.168110,-0.052688
industry,0.127473,1.000000,0.403638,0.202417,0.286591,0.238142,0.541939,0.227775
primary_extraction,-0.161388,0.403638,1.000000,0.182097,0.214073,0.011277,0.380886,0.389210
finance,0.194279,0.202417,0.182097,1.000000,0.335285,0.218055,0.325584,0.618937
information_communication,-0.162352,0.286591,0.214073,0.335285,1.000000,0.473612,0.450968,0.397145
public_sector,-0.192459,0.238142,0.011277,0.218055,0.473612,1.000000,0.168955,0.211852
business_services,-0.168110,0.541939,0.380886,0.325584,0.450968,0.168955,1.000000,0.319538
services,-0.052688,0.227775,0.389210,0.618937,0.397145,0.211852,0.319538,1.000000
